# PP-OCRv5 영문 인식기 파인튜닝 (Colab GPU)

절차 문서: `docs/파인튜닝_절차.md`. 이 노트북은 그 문서의 명령을 그대로 셀로 옮긴 것이다.

**준비물**: 로컬에서 `python notebooks/make_rec_dataset.py --blocks 6,7,9,10,14 --val-blocks 6 --out data/rec_train` 로 만든 `data/rec_train/` 을 검수(시트 확인)한 뒤 `rec_train.zip` 으로 묶어 Colab 에 업로드. (블록 1~5 는 절대 포함 금지)

**런타임**: GPU (T4 이상). 런타임 → 런타임 유형 변경 → GPU.

In [ ]:
!nvidia-smi | head -15
import subprocess, re
cuda = re.search(r"CUDA Version: (\d+)\.(\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA", cuda.groups() if cuda else "?")

## 1. 설치 (paddlepaddle-gpu 는 CUDA 버전에 맞춰 인덱스를 고른다)

In [ ]:
# CUDA 12.x 면 cu126, 11.8 이면 cu118. 위 셀 출력 확인 후 필요하면 바꿀 것.
# paddlepaddle.org.cn 인덱스가 Colab 에서 자주 타임아웃 → 휠 파일을 Baidu 저장소(bcebos)에서 직접 받는다. CUDA 11.8 이면 경로의 cu126 을 cu118 로.
import sys; v = f"cp{sys.version_info.major}{sys.version_info.minor}"
!pip install -q https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.3.1-{v}-{v}-linux_x86_64.whl
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -q -r requirements.txt
import paddle; print("paddle", paddle.__version__, "gpu:", paddle.is_compiled_with_cuda())

## 2. 데이터와 사전학습 가중치

In [ ]:
from google.colab import files
import os, zipfile
os.makedirs("train_data", exist_ok=True)
if not os.path.exists("train_data/rec_train/train_list.txt"):
    up = files.upload()                      # rec_train.zip 선택
    zf = [k for k in up if k.endswith(".zip")][0]
    with zipfile.ZipFile(zf) as z: z.extractall("train_data")
    # zip 안에 rec_train/ 폴더가 없으면 폴더로 감싼다
    if not os.path.exists("train_data/rec_train"):
        os.makedirs("train_data/rec_train", exist_ok=True)
        for n in ("imgs", "train_list.txt", "val_list.txt", "manifest.csv"):
            if os.path.exists(f"train_data/{n}"): os.rename(f"train_data/{n}", f"train_data/rec_train/{n}")
print("train", sum(1 for _ in open("train_data/rec_train/train_list.txt", encoding="utf-8")),
      "val", sum(1 for _ in open("train_data/rec_train/val_list.txt", encoding="utf-8")))
!head -3 train_data/rec_train/train_list.txt
!wget -q -nc https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la en_PP-OCRv5_mobile_rec_pretrained.pdparams

In [ ]:
# (선택) 도트매트릭스 합성 데이터 합치기. 로컬에서 `python notebooks/make_synth_dotmatrix.py --n 3000` 로 만든 data/rec_synth 를
# rec_synth.zip 으로 올린 뒤 실행. 합성 크롭은 train 에만 넣고 val 은 실제 크롭만 유지한다 (val 로 합성 과적합을 감지).
import os, zipfile
if os.path.exists("/content/rec_synth.zip") and not os.path.exists("train_data/rec_synth"):
    with zipfile.ZipFile("/content/rec_synth.zip") as z: z.extractall("train_data/rec_synth")
if os.path.exists("train_data/rec_synth/synth_list.txt"):
    lines = [l for l in open("train_data/rec_synth/synth_list.txt", encoding="utf-8") if l.strip()]
    with open("train_data/rec_train/train_list.txt", "a", encoding="utf-8") as f:
        for l in lines:
            fn, lab = l.rstrip("
").split("	"); f.write(f"../rec_synth/{fn}	{lab}
")   # data_dir 기준 상대 경로
    print("합성", len(lines), "장 추가 → train 총", sum(1 for _ in open("train_data/rec_train/train_list.txt", encoding="utf-8")))
else:
    print("합성 데이터 없음 (실제 크롭만으로 학습)")

## 3. 설정 파일 생성 (공식 yaml 을 읽어 우리 값으로 덮어쓴다)

In [ ]:
import yaml
SRC = "configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml"
cfg = yaml.safe_load(open(SRC, encoding="utf-8"))
EPOCHS = 30          # 데이터 수백 장 기준. val acc 가 정체하면 줄인다
LR = 0.0001          # 기본 0.0005 의 1/5 (파인튜닝)
cfg["Global"].update({
    "epoch_num": EPOCHS, "save_model_dir": "./output/itda_en_rec",
    "pretrained_model": "./en_PP-OCRv5_mobile_rec_pretrained.pdparams",
    "eval_batch_step": [0, 100], "save_epoch_step": 5, "print_batch_step": 20,
    "distributed": False, "use_gpu": True,
})
cfg["Optimizer"]["lr"]["learning_rate"] = LR
cfg["Train"]["dataset"]["data_dir"] = "./train_data/rec_train/"
cfg["Train"]["dataset"]["label_file_list"] = ["./train_data/rec_train/train_list.txt"]
cfg["Train"]["loader"]["batch_size_per_card"] = 32
# 공식 yaml 은 MultiScaleSampler.first_bs(128) 가 실제 배치를 정한다. 여기를 안 줄이면 T4(15GB) 에서 OOM (9/11 실측).
cfg["Train"]["sampler"]["first_bs"] = 32
cfg["Train"]["sampler"]["fix_bs"] = True
cfg["Train"]["loader"]["num_workers"] = 2
cfg["Eval"]["loader"]["num_workers"] = 2
cfg["Eval"]["dataset"]["data_dir"] = "./train_data/rec_train/"
cfg["Eval"]["dataset"]["label_file_list"] = ["./train_data/rec_train/val_list.txt"]
cfg["Eval"]["loader"]["batch_size_per_card"] = 64
# 공식 yaml 은 배치 크기를 앵커(&bs)로 공유한다. 덤프하면 앵커가 풀리므로 그대로 두어도 된다.
os.makedirs("configs/rec/itda", exist_ok=True)
yaml.safe_dump(cfg, open("configs/rec/itda/itda_en_rec.yaml", "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(open("configs/rec/itda/itda_en_rec.yaml", encoding="utf-8").read()[:1200])

## 4. 학습 → 평가 → 내보내기

In [ ]:
!python tools/train.py -c configs/rec/itda/itda_en_rec.yaml 2>&1 | grep -v "^$" | tail -40

In [ ]:
!python tools/eval.py -c configs/rec/itda/itda_en_rec.yaml -o Global.pretrained_model=./output/itda_en_rec/best_accuracy.pdparams 2>&1 | tail -5
# 비교 기준: 파인튜닝 전 사전학습 모델의 val 성능
!python tools/eval.py -c configs/rec/itda/itda_en_rec.yaml -o Global.pretrained_model=./en_PP-OCRv5_mobile_rec_pretrained.pdparams 2>&1 | tail -5

In [ ]:
!python tools/export_model.py -c configs/rec/itda/itda_en_rec.yaml     -o Global.pretrained_model=./output/itda_en_rec/best_accuracy.pdparams Global.save_inference_dir=./itda_en_rec_infer/
!ls -la itda_en_rec_infer
!cd itda_en_rec_infer && zip -q -r ../itda_en_rec_infer.zip . && cd .. && ls -la itda_en_rec_infer.zip
from google.colab import files
files.download("itda_en_rec_infer.zip")

## 5. 로컬에서 교체하고 500장에서 확인

1. `weights/en_PP-OCRv5_mobile_rec/` 를 백업하고 zip 의 `inference.json`(또는 `.pdmodel`), `inference.pdiparams`, `inference.yml` 을 그 폴더에 덮어쓴다. `config.json` 은 기존 것 유지.
2. `ITDA_INPUT_DIR=./val500 ITDA_OUTPUT_PATH=./pred500_ft.csv jupyter nbconvert --to notebook --execute predict.ipynb --output /tmp/run.ipynb`
3. `python notebooks/eval.py --pred pred500_ft.csv --blocks 1-5 --out eval500_ft.csv` → **73.6% 와 비교. 회복·퇴보 수를 따로 본다.**
4. 채택하면 파인튜닝 가중치를 GitHub Release 나 공개 링크에 올리고 `download_weights.sh` 가 받도록 바꾼다.